# Answer-flip study — Step 1: port and gate the compressors (Run-All)

**Purpose.** Before the main study, check that every compressor we ported behaves like its published version, and test the two noise-floor passes. Uses only the **pilot** Video-MME chunks (01, 05, 10, 15, 19), which are excluded from the confirmatory analysis.

| Method | Settings | Code followed |
|---|---|---|
| FastVID (anchor, already validated) | 10%, 25% | official LLaVA-OV code |
| VisionZip | 5%, 10%, 15%, 25% | official LLaVA-OV port in the HoliTom repo |
| DyCoke (TTM) | K = 0.9, K = 1.0 (its lowest budgets) | official DyCoke code |
| Uniform pooling | 5%, 10%, 25% | baseline |
| Random drop | 5%, 10%, 25% | baseline |

Plus the full-token pass and two **noise-floor** passes of the full model: frames sampled half an interval later, and answer options cyclically reordered (mapped back before comparing).

**Gate (written to `gate_rules.json` before any data):** a method/setting passes if its Video-MME relative accuracy on these questions is within 2 points of a published value for the same setting on LLaVA-OV-7B, or its 95% video-clustered CI covers that value. Settings without a published value (5%, baselines) are reported, not gated.

## How to run
1. Settings → Accelerator **GPU T4 x2**, Internet **On**, HF_TOKEN in Add-ons → Secrets (already set).
2. `QUICK_TEST = True` → **Run All** (~20 min). Check: `Compressor self-test OK ✅`, `Attention preflight OK ✅`, equivalence check `PASS`, a few ✓ lines.
3. `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**. Expect ~6–7 h for ~600 questions.
4. Send back `summary.json`, `gate_table.csv`, `step1.png`, `gate_rules.json`, `equivalence_check.json` (+ `errors.log` if not empty).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CONFIG — only QUICK_TEST should be changed                       ║
# ╚══════════════════════════════════════════════════════════════════╝
QUICK_TEST = True   # True = ~20-min smoke test with a tiny model. False = the real Step 1 run.

MODEL_ID_REAL  = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
MODEL_ID_QUICK = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
SIGLIP_ID      = "google/siglip-so400m-patch14-384"   # FastVID uses the original SigLIP pooling head

NUM_FRAMES      = 32
VIDEOMME_CHUNKS = ["01", "05", "10", "15", "19"]     # PILOT chunks only (never used in the confirmatory study)
MAX_Q_PER_CHUNK = None

FASTVID = dict(c=8, tau=0.9, d=0.4, p=4, beta=0.6)   # official LLaVA-OV defaults
METHOD_GRID = {
    "fastvid":   [0.10, 0.25],
    "visionzip": [0.05, 0.10, 0.15, 0.25],
    "dycoke":    [0.9, 1.0],                # TTM pruning rate K (its lowest budgets)
    "uniform":   [0.05, 0.10, 0.25],
    "random":    [0.05, 0.10, 0.25],
}
NOISE_FRAME_SHIFT = 0.5   # fraction of one sampling interval

# Published Video-MME scores on LLaVA-OV-7B (32 frames), as relative accuracy in % of the vanilla 58.6.
# FastVID paper Table 1 (arXiv 2503.11187) and HoliTom paper Table 2 (arXiv 2505.21334).
GATE_REFS = {
    ("fastvid", "0.1"):    {"values": [97.8], "src": "FastVID T1, 9.7%: 57.3"},
    ("fastvid", "0.25"):   {"values": [99.0], "src": "FastVID T1, 25%: 58.0"},
    ("visionzip", "0.1"):  {"values": [90.1, 91.1], "src": "FastVID T1 VisionZip* 9.7%: 52.8; HoliTom T2 10%: 53.4"},
    ("visionzip", "0.15"): {"values": [94.7, 95.7], "src": "FastVID T1 VisionZip* 14.8%: 55.5; HoliTom T2 15%: 56.1"},
    ("visionzip", "0.25"): {"values": [99.0, 99.3], "src": "FastVID T1 VisionZip* 25%: 58.0; HoliTom T2 25%: 58.2"},
    ("dycoke", "K0.9"):    {"values": [97.4], "src": "FastVID T1 DyCoke 32.5%: 57.1"},
    ("dycoke", "K1.0"):    {"values": [87.0, 92.7], "src": "FastVID T1 DyCoke 25%: 51.0; HoliTom T2 25%: 54.3"},
}
GATE_TOL_POINTS = 2.0

SEED = 0
MAX_RUNTIME_HOURS = 10.5
OUT_DIR  = "/kaggle/working/flips_step1"
WORK_DIR = "/tmp/astra_data"

if QUICK_TEST:
    MODEL_ID = MODEL_ID_QUICK
    NUM_FRAMES = 16
    VIDEOMME_CHUNKS = ["01"]
    MAX_Q_PER_CHUNK = 12
    OUT_DIR = OUT_DIR + "_quick"
else:
    MODEL_ID = MODEL_ID_REAL
print("Mode:", "QUICK TEST" if QUICK_TEST else "REAL STEP 1 RUN", "| model:", MODEL_ID, "| frames:", NUM_FRAMES)


### 1 · Setup

In [ ]:
# ── Setup: folders, environment, small installs ──────────────────────
import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf_home")      # keep HF cache on the big temp disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "WARN ") + "pip install " + " ".join(pkgs))

pip_install("hf_xet")   # faster Hugging Face downloads (harmless if already present)
pip_install("av")       # backup video decoder

try:
    from transformers import LlavaOnevisionForConditionalGeneration  # noqa: F401
    print("transformers OK")
except ImportError:
    pip_install("-U", "transformers", "accelerate")
    raise SystemExit("transformers was upgraded -> menu 'Run > Restart & Clear Cell Outputs', then 'Run All' again.")


### 1b · Gate rule and published numbers (written before any data)

In [ ]:
# ── Record the gate rule and reference numbers before any data is touched ──
import json, datetime, os
os.makedirs(OUT_DIR, exist_ok=True)
GATE_RECORD = {
    "written_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model": MODEL_ID, "frames": NUM_FRAMES, "chunks": VIDEOMME_CHUNKS,
    "rule": (f"A (method, setting) passes if its relative accuracy (method acc / full-token acc on the same questions) "
             f"is within {GATE_TOL_POINTS} points of a published value, or its 95% video-clustered bootstrap CI "
             f"covers a published value. Failing methods are excluded from the main study and reported."),
    "references_percent": {f"{m}@{s}": v for (m, s), v in GATE_REFS.items()},
    "methods": METHOD_GRID, "noise_floor": {"frame_shift": NOISE_FRAME_SHIFT, "option_order": "cyclic shift by one"},
}
path = os.path.join(OUT_DIR, "gate_rules.json")
if not os.path.exists(path):
    with open(path, "w") as f:
        json.dump(GATE_RECORD, f, indent=2)
print(json.dumps(GATE_RECORD, indent=2))


### 2 · Imports and GPU check

In [ ]:
# ── Imports + hardware check ──────────────────────────────────────────
import json, math, time, random, shutil, zipfile, gc, heapq
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from huggingface_hub import hf_hub_download, list_repo_files
import transformers
from transformers import LlavaOnevisionForConditionalGeneration, AutoTokenizer

print("torch", torch.__version__, "| transformers", transformers.__version__)
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, "No GPU found! Right panel > Settings > Accelerator > 'GPU T4 x2', then Run All again."
for i in range(N_GPU):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}")
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print(f"Free disk for temp videos: {free_gb:.1f} GB")
if free_gb < 12:
    print("⚠️ Less than 12 GB free — a 5 GB video chunk needs ~11 GB while unzipping.")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Optional: a Hugging Face token stored as a Kaggle Secret named HF_TOKEN (Add-ons > Secrets).
# Not needed for these public repos, but harmless.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception:
    print("No HF_TOKEN secret (fine for public repos)")


### 3 · Video-MME loader (+ half-interval frame shift)

In [ ]:
# ── Data: Video-MME questions + video chunks + frame reading ────────────
VMME_REPOS = ["lmms-eval/Video-MME", "lmms-lab/Video-MME"]

def _pick_repo():
    for repo in VMME_REPOS:
        try:
            return repo, list_repo_files(repo, repo_type="dataset")
        except Exception as e:
            print(f"Repo {repo} not reachable: {type(e).__name__}: {e}")
    raise RuntimeError("Cannot reach Video-MME on Hugging Face. Is Internet ON? (Settings > Internet)")

VMME_REPO, VMME_FILES = _pick_repo()
print("Using", VMME_REPO, "|", len(VMME_FILES), "files")

def load_videomme_table():
    pq = [f for f in VMME_FILES if f.endswith(".parquet")]
    assert pq, "No question table (.parquet) found in the Video-MME repo"
    df = pd.concat([pd.read_parquet(hf_hub_download(VMME_REPO, f, repo_type="dataset", local_dir=WORK_DIR))
                    for f in pq], ignore_index=True)
    low = {c.lower(): c for c in df.columns}
    def col(*names):
        for n in names:
            if n in low:
                return low[n]
        return None
    c_key, c_vid = col("videoid", "video_id"), col("video_id", "videoid")
    c_q, c_opt, c_ans = col("question"), col("options"), col("answer")
    assert all([c_key, c_q, c_opt, c_ans]), f"Unexpected columns: {list(df.columns)}"
    c_qid, c_dur, c_task = col("question_id"), col("duration"), col("task_type")
    out = pd.DataFrame({
        "video_key": df[c_key].astype(str),
        "video_id":  df[c_vid].astype(str),
        "qid":       df[c_qid].astype(str) if c_qid else df.index.astype(str),
        "duration":  df[c_dur].astype(str) if c_dur else "unknown",
        "task_type": df[c_task].astype(str) if c_task else "unknown",
        "question":  df[c_q].astype(str),
        "options":   df[c_opt].apply(lambda o: [str(x) for x in list(o)]),
        "answer":    df[c_ans].astype(str).str.strip().str[0].str.upper(),
    })
    print(f"Video-MME table: {len(out)} questions, {out.video_key.nunique()} videos")
    return out

VIDEO_EXT = (".mp4", ".mkv", ".webm", ".avi", ".mov")

def fetch_chunk(chunk):
    """Download one ~5 GB zip, extract only the videos, delete the zip. Returns {file stem: path}."""
    fn = f"videos_chunked_{chunk}.zip"
    assert fn in VMME_FILES, f"{fn} not in repo"
    t0 = time.time()
    zpath = hf_hub_download(VMME_REPO, fn, repo_type="dataset", local_dir=WORK_DIR)
    vdir = os.path.join(WORK_DIR, f"videos_{chunk}")
    os.makedirs(vdir, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if m.filename.lower().endswith(VIDEO_EXT):
                with z.open(m) as src, open(os.path.join(vdir, os.path.basename(m.filename)), "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
    os.remove(zpath)
    shutil.rmtree(os.path.join(os.environ["HF_HOME"], "xet"), ignore_errors=True)
    vids = {os.path.splitext(f)[0]: os.path.join(vdir, f) for f in os.listdir(vdir)}
    print(f"Chunk {chunk}: {len(vids)} videos ready in {time.time()-t0:.0f}s "
          f"(free disk {shutil.disk_usage(WORK_DIR).free/1e9:.1f} GB)")
    return vids

def cleanup_chunk(chunk):
    shutil.rmtree(os.path.join(WORK_DIR, f"videos_{chunk}"), ignore_errors=True)

def _frame_indices(total, n, shift):
    """Uniform sampling as in LMMs-Eval (np.linspace over all frames). shift = fraction of one sampling
    interval to move every sample later (0.5 = the noise-floor 'half-interval frame shift')."""
    idx = np.linspace(0, total - 1, n)
    if shift:
        idx = idx + shift * (total - 1) / max(n - 1, 1)
    return np.clip(idx, 0, total - 1).round().astype(int)

def _read_cv2(path, n, shift=0.0):
    import cv2
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total > 0:
        for k in _frame_indices(total, n, shift):
            ok = False
            for off in (0, -1, 1, -5, 5, -20):
                kk = int(np.clip(k + off, 0, total - 1))
                cap.set(cv2.CAP_PROP_POS_FRAMES, kk)
                ok, fr = cap.read()
                if ok:
                    break
            if ok:
                frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def _read_av(path, n, shift=0.0):
    import av
    out = []
    with av.open(path) as c:
        s = c.streams.video[0]
        dur = float(s.duration * s.time_base) if s.duration else float(c.duration or 0) / 1e6
        end = max(dur - 0.5, 0)
        ts = np.linspace(0, end, n)
        if shift:
            ts = np.clip(ts + shift * end / max(n - 1, 1), 0, end)
        for t in ts:
            c.seek(int(t / s.time_base), stream=s, any_frame=False, backward=True)
            for fr in c.decode(s):
                out.append(fr.to_ndarray(format="rgb24"))
                break
    return out

def read_frames(path, n, shift=0.0):
    frames = []
    try:
        frames = _read_cv2(path, n, shift)
    except Exception as e:
        print("cv2 failed:", e)
    if len(frames) < n:
        try:
            frames = _read_av(path, n, shift)
        except Exception as e:
            print("av failed:", e)
    if len(frames) == 0:
        raise RuntimeError(f"Could not decode {path}")
    while len(frames) < n:          # pad rare short decodes
        frames.append(frames[-1])
    return frames[:n]


### 4 · Model + T4 memory fix + SigLIP heads for FastVID and VisionZip

In [ ]:
# ── Model: LLaVA-OneVision with direct control over visual tokens ──────
# ── T4 memory fix + safe attention routing ─────────────────────────────
# 1) Expand grouped K/V heads ourselves so PyTorch can use the memory-efficient kernel (math kernel -> OOM on T4).
# 2) Never hand an explicit mask to the memory-efficient kernel: on T4 that path can crash with
#    "illegal memory access". A plain causal mask is converted to is_causal=True (identical maths).
# 3) Guard: if any NON-causal mask (e.g. transformers mistaking position gaps for packed sequences)
#    reaches a long attention call, stop loudly instead of computing something silently wrong.
from torch.nn.attention import sdpa_kernel, SDPBackend
if not hasattr(torch.nn.functional, "_astra_orig_sdpa"):
    torch.nn.functional._astra_orig_sdpa = torch.nn.functional.scaled_dot_product_attention
_orig_sdpa = torch.nn.functional._astra_orig_sdpa
_TRIL_CACHE = {}
def _plain_causal(mask, L, S):
    if L != S or mask.dim() != 4 or mask.shape[0] != 1 or mask.shape[1] != 1:
        return False
    key = (L, str(mask.device))
    if key not in _TRIL_CACHE:
        _TRIL_CACHE.clear()
        _TRIL_CACHE[key] = torch.ones(L, L, dtype=torch.bool, device=mask.device).tril()
    allowed = mask[0, 0] if mask.dtype == torch.bool else (mask[0, 0] == 0)
    return torch.equal(allowed, _TRIL_CACHE[key])

def _sdpa_safe(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False, **kw):
    if enable_gqa and key.shape[-3] != query.shape[-3]:
        rep = query.shape[-3] // key.shape[-3]
        key = key.repeat_interleave(rep, dim=-3)
        value = value.repeat_interleave(rep, dim=-3)
    if attn_mask is not None:
        L, S = query.shape[-2], key.shape[-2]
        if _plain_causal(attn_mask, L, S):
            attn_mask, is_causal = None, True
        elif L > 1024:
            raise RuntimeError("A non-causal attention mask reached a long attention call — stopping instead of "
                               "computing silently wrong results. Send me this error.")
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale, **kw)

torch.nn.functional.scaled_dot_product_attention = _sdpa_safe
NO_MATH = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION]
print("T4 attention fix applied (K/V heads expanded, causal masks routed to is_causal, math kernel disabled for the LLM)")

max_memory = {i: "13GiB" for i in range(N_GPU)}
if N_GPU >= 2:
    max_memory[0] = "10GiB"
_dtype_kw = {"dtype": torch.float16} if int(transformers.__version__.split(".")[0]) >= 5 else {"torch_dtype": torch.float16}
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", max_memory=max_memory, attn_implementation="sdpa", **_dtype_kw).eval()
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def _get(root, paths):
    for p in paths:
        obj, ok = root, True
        for part in p.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError(f"None of {paths} found — send me the error and transformers version")

vision_tower = _get(model, ["vision_tower", "model.vision_tower"])
projector    = _get(model, ["multi_modal_projector", "model.multi_modal_projector"])
NEWLINE      = _get(model, ["image_newline", "model.image_newline"]).data
lm_head      = _get(model, ["lm_head", "language_model.lm_head"])
body = None
for _, mod in model.named_modules():
    if hasattr(mod, "embed_tokens") and hasattr(mod, "layers") and hasattr(mod, "norm"):
        body = mod
        break
assert body is not None, "Could not find the language decoder"

CFG, VCFG, TCFG = model.config, model.config.vision_config, model.config.text_config
VFL = CFG.vision_feature_layer
VSS = getattr(CFG, "vision_feature_select_strategy", "full")
IMG_SIZE, PATCH = VCFG.image_size, VCFG.patch_size
SIDE = IMG_SIZE // PATCH
POOLED = math.ceil(SIDE / 2)
NLAYERS, HID = TCFG.num_hidden_layers, TCFG.hidden_size
NHEADS, NKV = TCFG.num_attention_heads, TCFG.num_key_value_heads
HEAD_DIM = getattr(TCFG, "head_dim", None) or HID // NHEADS
INTER = TCFG.intermediate_size
EMB_DEVICE = body.embed_tokens.weight.device
LM_DEVICE = lm_head.weight.device
try:
    from transformers import AutoImageProcessor
    _ip = AutoImageProcessor.from_pretrained(MODEL_ID)
    IMG_MEAN, IMG_STD = list(_ip.image_mean), list(_ip.image_std)
except Exception:
    IMG_MEAN, IMG_STD = [0.5] * 3, [0.5] * 3

# ── FastVID needs the ORIGINAL SigLIP post-layernorm + attention-pooling head (as in the official code) ──
from transformers import SiglipVisionModel
_sig = SiglipVisionModel.from_pretrained(SIGLIP_ID, **_dtype_kw)
VDEV = next(vision_tower.parameters()).device
# transformers v4 nests these under .vision_model; v5 puts them directly on the model
SIG_POSTLN = _get(_sig, ["vision_model.post_layernorm", "post_layernorm", "model.post_layernorm"]).to(VDEV).eval()
SIG_HEAD = _get(_sig, ["vision_model.head", "head", "model.head"]).to(VDEV).eval()
for _need in ("probe", "attention", "layernorm", "mlp"):
    assert hasattr(SIG_HEAD, _need), (f"SigLIP head has no '{_need}' (type {type(SIG_HEAD).__name__}, "
                                      f"children {[n for n, _ in SIG_HEAD.named_children()]}) — send me this error")
assert isinstance(SIG_HEAD.attention, torch.nn.MultiheadAttention), \
    f"Unexpected SigLIP head attention type {type(SIG_HEAD.attention).__name__} — send me this error"
print("SigLIP pooling head loaded (post_layernorm + attention-pooling head) ✅")

# ── VisionZip needs the attention and keys of the vision layer that produces the features ──
# (official LLaVA-OV port in the HoliTom repo: attentions[-1].mean(heads).mean(queries) and keys.mean(heads))
ENC_LAYERS = _get(vision_tower, ["vision_model.encoder.layers", "encoder.layers", "model.encoder.layers"])
assert isinstance(VFL, int), "VisionZip port assumes a single vision feature layer"
_n_hs = len(ENC_LAYERS) + 1
_h_idx = VFL if VFL >= 0 else _n_hs + VFL
VZ_LAYER, VZ_IN = ENC_LAYERS[_h_idx - 1], _h_idx - 1
VZ_HEADS = VCFG.num_attention_heads
VZ_HD = VCFG.hidden_size // VZ_HEADS
print(f"VisionZip reads attention/keys from vision layer {_h_idx - 1} of {len(ENC_LAYERS)}")
del _sig; gc.collect()
print(f"LLM layers={NLAYERS} hidden={HID} heads={NHEADS}/{NKV} | {SIDE}x{SIDE} -> {POOLED}x{POOLED} tokens/frame")

LETTERS = "ABCDEFGH"
LETTER_IDS = []
for L in LETTERS:
    ids = tok.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"Letter {L} is not a single token"
    LETTER_IDS.append(ids[0])

PREFIX = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n"
PREFIX_IDS = tok(PREFIX, add_special_tokens=False, return_tensors="pt").input_ids[0]

def build_suffix_ids(question, options):
    opts = "\n".join(str(o) for o in options)
    s = ("\nSelect the best answer to the following multiple-choice question based on the video. "
         "Respond with only the letter (A, B, C, or D) of the correct option.\n"
         f"{question}\n{opts}\nThe best answer is:<|im_end|>\n<|im_start|>assistant\n")
    return tok(s, add_special_tokens=False, return_tensors="pt").input_ids[0]

def preprocess(frames):
    arr = np.stack([np.asarray(Image.fromarray(f).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))
                    for f in frames]).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(0, 3, 1, 2)
    m, s = torch.tensor(IMG_MEAN)[:, None, None], torch.tensor(IMG_STD)[:, None, None]
    return (x - m) / s

@torch.no_grad()
def encode_video(pix, bs=8):
    """frames -> (T, POOLED*POOLED, D) projected tokens (exactly LLaVA-OV's video path, before the newline token),
    FastVID's per-frame SigLIP-head attention (T, P') and global feature (T, 1152),
    and VisionZip's pooled vision attention (T, P') and pooled keys (T, P', head_dim)."""
    pdev = next(projector.parameters()).device
    feats, attns, globs, vz_a, vz_m = [], [], [], [], []
    for i in range(0, pix.shape[0], bs):
        out = vision_tower(pixel_values=pix[i:i+bs].to(VDEV, torch.float16), output_hidden_states=True)
        h = out.hidden_states[VFL]
        if VSS == "default":
            h = h[:, 1:]
        # FastVID "vision abstract": original SigLIP post_layernorm + attention-pooling head on the same features
        z = SIG_POSTLN(h.to(VDEV))
        probe = SIG_HEAD.probe.repeat(z.shape[0], 1, 1)
        att_out, att_w = SIG_HEAD.attention(probe, z, z, need_weights=True)      # att_w: (b, 1, SIDE*SIDE)
        pooled = att_out + SIG_HEAD.mlp(SIG_HEAD.layernorm(att_out))
        globs.append(pooled[:, 0].float())
        attns.append(att_w.float())
        # VisionZip: self-attention of the feature-producing layer, recomputed (sdpa does not return weights)
        xin = out.hidden_states[VZ_IN].to(VZ_LAYER.layer_norm1.weight.device)
        if VSS == "default":
            xin = xin[:, 1:]
        xin = VZ_LAYER.layer_norm1(xin)
        b, P, _ = xin.shape
        q = VZ_LAYER.self_attn.q_proj(xin).float().view(b, P, VZ_HEADS, VZ_HD).transpose(1, 2)
        k = VZ_LAYER.self_attn.k_proj(xin).float().view(b, P, VZ_HEADS, VZ_HD).transpose(1, 2)
        w = torch.softmax(torch.matmul(q, k.transpose(2, 3)) * VZ_HD ** -0.5, dim=-1)
        vz_a.append(w.mean(dim=1).mean(dim=1))          # (b, P): attention each token receives
        vz_m.append(k.mean(dim=1))                      # (b, P, head_dim): keys averaged over heads
        del q, k, w
        feats.append(projector(h.to(pdev)))
    def pool(t):                                        # (T, SIDE*SIDE, C) -> (T, POOLED*POOLED, C), = get_2dPool
        T, P, C = t.shape
        t = t.view(T, SIDE, SIDE, C).permute(0, 3, 1, 2).float()
        t = F.interpolate(t, size=(POOLED, POOLED), mode="bilinear")
        return t.permute(0, 2, 3, 1).reshape(T, POOLED * POOLED, C)
    f = pool(torch.cat(feats)).contiguous().to(torch.float16)
    dev = f.device
    a = pool(torch.cat(attns).view(-1, SIDE * SIDE, 1).to(dev))[..., 0]
    va = pool(torch.cat(vz_a).unsqueeze(-1).to(dev))[..., 0]
    vm = pool(torch.cat(vz_m).to(dev))
    return f, a, torch.cat(globs).to(dev), va, vm

def llm_flops(n):
    dkv = NKV * HEAD_DIM
    per_layer = 2 * (2 * n * HID * HID + 2 * n * HID * dkv + 2 * n * n * HID + 3 * n * HID * INTER)
    return float(NLAYERS * per_layer)

def _sync():
    for i in range(N_GPU):
        torch.cuda.synchronize(i)

@torch.no_grad()
def llm_pass(vis, vis_pos, n_video_full, suf_ids, n_opt, pre_ids=None):
    """One prefill. vis: kept video tokens (K, D); vis_pos: their ORIGINAL indices in [0, n_video_full)
    (FastVID keeps original position ids). The newline + question keep their original positions too."""
    E = body.embed_tokens
    pre_ids = PREFIX_IDS if pre_ids is None else pre_ids
    pre, suf = E(pre_ids.to(EMB_DEVICE)), E(suf_ids.to(EMB_DEVICE))
    x = torch.cat([pre, vis.to(EMB_DEVICE, pre.dtype), NEWLINE.to(EMB_DEVICE, pre.dtype)[None], suf], 0)[None]
    n_pre, n_q = pre.shape[0], suf.shape[0]
    vp = torch.as_tensor(vis_pos, device=EMB_DEVICE, dtype=torch.long)
    pos = torch.cat([torch.arange(n_pre, device=EMB_DEVICE), n_pre + vp,
                     torch.tensor([n_pre + n_video_full], device=EMB_DEVICE),
                     n_pre + n_video_full + 1 + torch.arange(n_q, device=EMB_DEVICE)])[None]
    # explicit all-ones mask: without it, newer transformers treats position gaps as "packed sequences"
    am = torch.ones(1, x.shape[1], dtype=torch.long, device=EMB_DEVICE)
    for i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(i)
    _sync(); t0 = time.time()
    with sdpa_kernel(NO_MATH):
        out = body(inputs_embeds=x, attention_mask=am, position_ids=pos, use_cache=False)
    h = out[0][0, -1]
    logits = lm_head(h.to(LM_DEVICE)[None])[0].float()
    _sync(); dt = time.time() - t0
    lp = torch.log_softmax(logits[LETTER_IDS[:n_opt]], -1).cpu().numpy()
    if not np.all(np.isfinite(lp)):
        raise FloatingPointError("NaN/inf logits (fp16 overflow)")
    srt = np.sort(lp)[::-1]
    return {"pred": LETTERS[int(lp.argmax())], "logp": lp.round(4).tolist(), "margin": float(srt[0] - srt[1]),
            "ntok": int(vis.shape[0]), "sec": float(dt), "flops": llm_flops(x.shape[1]),
            "peak_gb": round(max(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1e9, 3)}

@torch.no_grad()
def preflight_attention(n_tok):
    dev = body.layers[0].self_attn.q_proj.weight.device
    q = torch.randn(1, NHEADS, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    kv = torch.randn(1, NKV, n_tok, HEAD_DIM, device=dev, dtype=torch.float16)
    mask = torch.ones(n_tok, n_tok, dtype=torch.bool, device=dev).tril()[None, None]
    torch.cuda.reset_peak_memory_stats(dev)
    base = torch.cuda.memory_allocated(dev)
    try:
        with sdpa_kernel(NO_MATH):
            o1 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, is_causal=True, enable_gqa=True)
            o2 = torch.nn.functional.scaled_dot_product_attention(q, kv, kv, attn_mask=mask, enable_gqa=True)  # routed to is_causal
        torch.cuda.synchronize(dev)          # surface any asynchronous CUDA error HERE, not in a later cell
        assert torch.equal(o1, o2), "causal-mask routing changed the result"
    except RuntimeError as e:
        raise RuntimeError(f"Memory-efficient attention is not available here ({e}). Send me this message.")
    del o1, o2
    peak = (torch.cuda.max_memory_allocated(dev) - base) / 1e9
    del q, kv, mask
    torch.cuda.empty_cache()
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU{i}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
    print(f"Attention preflight OK ✅  {n_tok} tokens -> extra peak memory {peak:.2f} GB")

preflight_attention(NUM_FRAMES * POOLED ** 2 + 400)
print("Model ready ✅")


### 4b · Safety check vs the official model forward (runs on the first video)

In [ ]:
# ── Safety check: our token pipeline must reproduce the official model forward ──
from transformers import AutoProcessor

def equivalence_check(frames, row):
    report = {"transformers": transformers.__version__}
    try:
        proc = AutoProcessor.from_pretrained(MODEL_ID)
        opts = "\n".join(str(o) for o in row.options)
        text = ("Select the best answer to the following multiple-choice question based on the video. "
                "Respond with only the letter (A, B, C, or D) of the correct option.\n"
                f"{row.question}\n{opts}\nThe best answer is:")
        conv = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": text}]}]
        prompt = proc.apply_chat_template(conv, add_generation_prompt=True)
        inputs = proc(text=prompt, videos=[np.stack(frames)], return_tensors="pt")
        inputs = {k: (v.to(EMB_DEVICE, torch.float16) if torch.is_floating_point(v) else v.to(EMB_DEVICE))
                  for k, v in inputs.items()}
        n_opt = min(len(row.options), len(LETTERS))
        with torch.no_grad(), sdpa_kernel(NO_MATH):
            try:
                out = model(**inputs, logits_to_keep=1)
            except TypeError:
                out = model(**inputs, num_logits_to_keep=1)
        lp_off = torch.log_softmax(out.logits[0, -1].float()[LETTER_IDS[:n_opt]], -1).cpu().numpy()
        del out
        ids = inputs["input_ids"][0].cpu()
        vid_tok = getattr(CFG, "video_token_id", None) or getattr(CFG, "video_token_index", None)
        posi = (ids == vid_tok).nonzero().flatten()
        X = encode_video(inputs["pixel_values_videos"][0])[0]
        vis = X.reshape(-1, X.shape[-1])
        report["n_video_tokens_official"] = int(len(posi))
        report["n_video_tokens_ours"] = int(vis.shape[0] + 1)
        ours = llm_pass(vis, np.arange(vis.shape[0]), vis.shape[0], ids[posi[-1] + 1:], n_opt, pre_ids=ids[: posi[0]])
        lp_ours = np.array(ours["logp"])
        report["logp_official"] = lp_off.round(4).tolist()
        report["logp_ours_same_pixels"] = lp_ours.round(4).tolist()
        report["max_abs_gap"] = float(np.abs(lp_off - lp_ours).max())
        report["pixel_mean_abs_diff_ours_vs_official"] = float(
            (preprocess(frames) - inputs["pixel_values_videos"][0].float().cpu()).abs().mean())
        ok = (report["n_video_tokens_official"] == report["n_video_tokens_ours"] and report["max_abs_gap"] <= 0.15)
        report["status"] = "PASS" if ok else "FAIL"
    except Exception as e:
        report["status"] = "SKIPPED"
        report["error"] = f"{type(e).__name__}: {e}"
    with open(os.path.join(OUT_DIR, "equivalence_check.json"), "w") as f:
        json.dump(report, f, indent=2)
    print("Equivalence check:", json.dumps(report, indent=2))
    if report["status"] == "FAIL" or (report["status"] == "SKIPPED" and not QUICK_TEST):
        raise RuntimeError("Equivalence check failed/skipped — stopping so no GPU hours are wasted. "
                           "Send me equivalence_check.json and the log.")
    gc.collect(); torch.cuda.empty_cache()


### 5 · Compressors: FastVID, DyCoke, VisionZip, uniform pooling, random drop (+ self-test)

In [ ]:
# ══ Compressors ══
# ── FastVID (Shen et al., NeurIPS 2025) — faithful port of the official LLaVA-OneVision code ──
# Source: github.com/LunarShen/FastVID, fastvid_llavaonevision/LLaVA-NeXT/llava/model/language_model/modeling_qwen2.py
# (MIT License, Copyright (c) 2025 Shen Leqi). Logic is unchanged; the only edits are
# (1) a seeded generator for the 1e-6 tie-break noise (reproducibility) and
# (2) the token-merging sums are done in fp32 instead of fp16 to avoid overflow on long segments, and
# (3) broadcasting matmul instead of an expanded bmm (same numbers, far less memory).
@torch.no_grad()
def fastvid_compress(X, attn, gfeat, r, c=8, tau=0.9, d=0.4, p=4, beta=0.6, seed=0):
    """X: (T, L, D) projected video tokens; attn: (T, L) SigLIP-head attention; gfeat: (T, Dg) pooled features.
    Returns kept tokens (K, D) and their original indices in [0, T*L), sorted."""
    T, L, D = X.shape
    dev = X.device
    gen = torch.Generator(device=dev).manual_seed(seed)
    batchframe_indices = torch.arange(T, device=dev).unsqueeze(1)
    alltoken_indices = torch.arange(T * L, device=dev).view(T, L)
    attn = attn.to(dev)
    gfeat = gfeat.to(dev)
    # ── DySeg ──
    g = gfeat / gfeat.norm(dim=1, keepdim=True)
    sim = (g[:-1] * g[1:]).sum(dim=1)
    cut_topk = torch.topk(sim, min(c - 1, T - 1), largest=False).indices
    cut_cos = torch.nonzero(sim < tau, as_tuple=False).squeeze(1)
    cut = torch.unique(torch.cat([cut_topk, cut_cos])).sort().values
    padded = F.pad(cut, (1, 1), value=-1)
    padded[-1] = T - 1
    segment_sizes = padded.diff().tolist()
    # ── STPrune ──
    final_tokens, keep = [], []
    frame_retain_num = int(L * r)
    frame_salient_num = frame_retain_num - int(frame_retain_num * d)
    frame_context_num = frame_retain_num - frame_salient_num
    frm_context_num_list = torch.zeros(T, dtype=torch.int, device=dev)
    offset = 0
    for seg_len in segment_sizes:
        temp_num = (seg_len + p - 1) // p
        cur = (frame_context_num * seg_len) // temp_num
        seg_idx = torch.arange(seg_len - 1, -1, -1, device=dev)
        sub = frm_context_num_list[offset:offset + seg_len]
        sub[seg_idx % p == 0] = cur
        offset += seg_len
    # ── ATS: attention-based salient tokens ──
    salient_indexes = torch.topk(attn, frame_salient_num, dim=1).indices
    bi = batchframe_indices.expand(-1, frame_salient_num)
    final_tokens.append(X[bi, salient_indexes].reshape(-1, D))
    keep.append(alltoken_indices[bi, salient_indexes].reshape(-1))
    # ── density scores on the remaining tokens ──
    all_indices = torch.arange(L, device=dev).unsqueeze(0).expand(T, -1)
    m = torch.ones_like(all_indices, dtype=torch.bool)
    m.scatter_(1, salient_indexes, False)
    nf = L - frame_salient_num
    filtered_indices = all_indices[m].view(T, nf)
    bi = batchframe_indices.expand(-1, nf)
    token_filtered = X[bi, filtered_indices]
    all_filtered_idx = alltoken_indices[bi, filtered_indices]
    dist = torch.cdist(token_filtered.float(), token_filtered.float()) / (D ** 0.5)
    dist_nearest, _ = torch.topk(dist, k=4, dim=-1, largest=False)
    density = (-(dist_nearest ** 2).mean(dim=-1)).exp()
    density = density + torch.rand(density.shape, device=dev, dtype=density.dtype, generator=gen) * 1e-6
    dmask = (density[:, None, :] > density[:, :, None]).float()
    dist_max = dist.flatten(1).max(dim=-1)[0][:, None, None]
    dist_0, _ = (dist * dmask + dist_max * (1 - dmask)).min(dim=-1)
    density_score = dist_0 * density
    sampled = torch.topk(density_score, k=frame_context_num, dim=-1).indices
    tf32 = token_filtered.float()
    # ── DTM for single-frame segments ──
    bi = batchframe_indices.expand(-1, frame_context_num)
    ctx = tf32[bi, sampled]
    ctx_idx = all_filtered_idx[bi, sampled]
    tbm = tf32 / tf32.norm(dim=-1, keepdim=True)
    mt = tbm[bi, sampled]
    simm = torch.bmm(tbm, mt.transpose(1, 2))
    a1 = torch.zeros(T, nf, frame_context_num, device=dev)
    a1.scatter_(2, simm.argmax(dim=2).unsqueeze(-1), 1)
    aw = (1 / (a1.sum(dim=1).unsqueeze(-1) + 1)).clamp(min=beta)
    cnt = a1.sum(dim=1).clamp(min=1).unsqueeze(-1)
    agg = torch.bmm(a1.transpose(1, 2), tf32) / cnt
    ctx = aw * ctx + (1 - aw) * agg
    single = frm_context_num_list == frame_context_num
    final_tokens.append(ctx[single].reshape(-1, D).to(X.dtype))
    keep.append(ctx_idx[single].reshape(-1))
    # ── DTM for multi-frame segments ──
    start = 0
    for seg_len in segment_sizes:
        if seg_len > 1:
            lst = frm_context_num_list[start:start + seg_len]
            k_ctx = int(lst[-1])
            tmask = lst > frame_context_num
            tnum = int(tmask.sum())
            seg_ds = density_score[start:start + seg_len][tmask]
            seg_tok = tf32[start:start + seg_len]
            seg_target = seg_tok[tmask]
            seg_flat = seg_tok.reshape(-1, D)                    # all filtered tokens of the segment (shared by every anchor)
            seg_idx = all_filtered_idx[start:start + seg_len][tmask]
            smp = torch.topk(seg_ds, k=k_ctx, dim=-1).indices
            bj = batchframe_indices[:tnum].expand(-1, k_ctx)
            cctx = seg_target[bj, smp]
            cidx = seg_idx[bj, smp]
            tb = seg_flat / seg_flat.norm(dim=-1, keepdim=True)
            mtt = cctx / cctx.norm(dim=-1, keepdim=True)
            s2 = torch.matmul(tb[None], mtt.transpose(1, 2))      # = official bmm with the expanded segment
            a2 = torch.zeros(tnum, tb.shape[0], k_ctx, device=dev)
            a2.scatter_(2, s2.argmax(dim=2).unsqueeze(-1), 1)
            aw2 = (1 / (a2.sum(dim=1).unsqueeze(-1) + 1)).clamp(min=beta)
            cnt2 = a2.sum(dim=1).clamp(min=1).unsqueeze(-1)
            agg2 = torch.matmul(a2.transpose(1, 2), seg_flat) / cnt2
            cctx = aw2 * cctx + (1 - aw2) * agg2
            final_tokens.append(cctx.reshape(-1, D).to(X.dtype))
            keep.append(cidx.reshape(-1))
        start += seg_len
    toks = torch.cat(final_tokens, 0)
    idx = torch.cat(keep, 0)
    order = torch.argsort(idx)
    return toks[order], idx[order]


# ── DyCoke (Tao et al., CVPR 2025) — temporal token merging (TTM), faithful port ──
# Source: github.com/KD-TAO/DyCoke, llava/model/llava_arch.py `dycole_ttm` (Apache-2.0). Applied, as in the official
# code, to the flattened pooled projected tokens; kept tokens get contiguous positions. Stage 2 (KV-cache pruning)
# only runs at decoding steps (position_ids length 1), so it does not affect a one-token answer.
# The official loop structure is kept exactly, including that frame 30 of 32 stays unpruned.
@torch.no_grad()
def dycoke_ttm(X, K):
    T, L, D = X.shape
    feat = X.reshape(T * L, D)
    keep_frac = 1 - K
    n_keep = int(keep_frac * L)
    fr = lambda j: feat[j * L:(j + 1) * L]
    sims = []
    for i in range(0, T - 1, 2):
        a = F.normalize(fr(i).float(), p=2, dim=1); b = F.normalize(fr(i + 1).float(), p=2, dim=1)
        sims.append(F.cosine_similarity(a, b, dim=1))
    out = []
    for i in range(0, T - 1, 2):
        keep = sims[i // 2].topk(n_keep, largest=False).indices
        out.append(fr(i)); out.append(fr(i + 1)[keep])
    odd = []
    for i in range(0, T - 4, 4):
        odd.append(F.cosine_similarity(fr(i).float(), fr(i + 2).float(), dim=1))
    for i in range(0, T - 4, 4):
        keep = odd[i // 4].topk(n_keep, largest=False).indices
        out[i] = fr(i); out[i + 2] = fr(i + 2)[keep]
    return torch.cat(out, 0)

# ── VisionZip (Yang et al., CVPR 2025) — LLaVA-OV video port from the HoliTom repo (visionzip/llava_arch.py) ──
# Per frame, after LLaVA-OV's 2x2 pooling: dominant tokens by received vision attention, contextual tokens by merging
# the rest onto evenly spaced targets using key similarity. Default dominant ratio 54/64 (as in that code).
# Output order per frame = [dominant (spatial order), contextual]; frames concatenated; contiguous positions.
@torch.no_grad()
def visionzip_compress(X, vz_attn, vz_metric, r, dominant_ratio=54 / 64):
    T, S, D = X.shape
    spatial = int(S * r)
    dom = int(spatial * dominant_ratio)
    ctx = spatial - dom
    hs = X.float()
    att = vz_attn.to(hs.device).float()
    met = vz_metric.to(hs.device).float()
    topk = att.topk(dom, dim=1).indices
    mask = torch.ones(T, S, dtype=torch.bool, device=hs.device).scatter_(1, topk, False)   # True = not dominant
    dominant = hs.masked_select(~mask.unsqueeze(-1)).view(T, dom, D)
    met_f = met[mask].view(T, S - dom, met.shape[2])
    hs_f = hs.masked_select(mask.unsqueeze(-1)).view(T, S - dom, D)
    met_n = met_f / met_f.norm(dim=-1, keepdim=True)
    step = max(1, (S - dom) // ctx)
    tgt = torch.arange(0, S - dom, step, device=hs.device)[:ctx]
    tgt_tok = met_n[:, tgt, :]
    rest = ~torch.isin(torch.arange(met_n.shape[1], device=hs.device), tgt)
    to_merge = met_n[:, rest, :]
    sim = torch.bmm(to_merge, tgt_tok.transpose(1, 2))
    onehot = torch.zeros(T, to_merge.shape[1], ctx, device=hs.device)
    onehot.scatter_(2, sim.argmax(dim=2).unsqueeze(-1), 1)
    counts = onehot.sum(dim=1).clamp(min=1).unsqueeze(-1)
    agg = torch.bmm(onehot.transpose(1, 2), hs_f[:, rest, :]) / counts
    contextual = hs_f[:, tgt, :] + agg
    return torch.cat([dominant, contextual], dim=1).reshape(-1, D).to(X.dtype)

# ── Baselines ──
def _best_grid(n, side):
    best = None
    for h in range(1, side + 1):
        for w in range(1, side + 1):
            key = (abs(h * w - n), abs(h - w), -h * w)
            if best is None or key < best[0]:
                best = (key, h, w)
    return best[1], best[2]

@torch.no_grad()
def uniform_pool(X, r):
    """Per-frame average pooling of the 14x14 grid to the grid closest to int(196*r) tokens."""
    T, S, D = X.shape
    side = int(round(S ** 0.5))
    h, w = _best_grid(int(S * r), side)
    g = X.float().view(T, side, side, D).permute(0, 3, 1, 2)
    g = F.adaptive_avg_pool2d(g, (h, w))
    return g.permute(0, 2, 3, 1).reshape(-1, D).to(X.dtype)

@torch.no_grad()
def random_drop(X, r, seed):
    """Per frame, keep int(196*r) uniformly random tokens in spatial order."""
    T, S, D = X.shape
    k = int(S * r)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.stack([torch.randperm(S, generator=gen)[:k].sort().values for _ in range(T)]).to(X.device)
    return torch.gather(X, 1, idx.unsqueeze(-1).expand(-1, -1, D)).reshape(-1, D)

def compress_all(X, A, G, VA, VM, seed):
    """Every (method, setting) -> dict(tokens, pos, n_full). 'pos' are the position ids given to the LLM:
    FastVID keeps original positions (official code); all others use contiguous positions (official code / LLaVA default)."""
    T, L, D = X.shape
    out = {}
    def contig(t):
        return {"tokens": t, "pos": np.arange(t.shape[0]), "n_full": int(t.shape[0])}
    for r in METHOD_GRID["fastvid"]:
        t, i = fastvid_compress(X, A, G, r, seed=seed, **FASTVID)
        out[("fastvid", str(r))] = {"tokens": t, "pos": i.cpu().numpy(), "n_full": T * L}
    for K in METHOD_GRID["dycoke"]:
        out[("dycoke", f"K{K}")] = contig(dycoke_ttm(X, K))
    for r in METHOD_GRID["visionzip"]:
        out[("visionzip", str(r))] = contig(visionzip_compress(X, VA, VM, r))
    for r in METHOD_GRID["uniform"]:
        out[("uniform", str(r))] = contig(uniform_pool(X, r))
    for r in METHOD_GRID["random"]:
        out[("random", str(r))] = contig(random_drop(X, r, seed))
    return out

# ── self-test on random data (shapes, finiteness, token budgets) ──
_T, _L = NUM_FRAMES, POOLED * POOLED
_X = torch.randn(_T, _L, 64, device=EMB_DEVICE, dtype=torch.float16)
_A = torch.rand(_T, _L, device=EMB_DEVICE)
_G = torch.randn(_T, 32, device=EMB_DEVICE)
_VA = torch.rand(_T, _L, device=EMB_DEVICE)
_VM = torch.randn(_T, _L, 16, device=EMB_DEVICE)
for (m, s), c in compress_all(_X, _A, _G, _VA, _VM, seed=0).items():
    t = c["tokens"]
    assert torch.isfinite(t.float()).all(), (m, s)
    assert len(c["pos"]) == t.shape[0] and int(np.max(c["pos"])) < c["n_full"]
    if m == "fastvid":
        assert len(np.unique(c["pos"])) == len(c["pos"]) and np.all(np.diff(c["pos"]) > 0)
    print(f"  {m:9s} {s:5s} kept {t.shape[0]:5d} tokens = {t.shape[0] / (_T * _L):.3f}")
del _X, _A, _G, _VA, _VM
print("Compressor self-test OK ✅")


### 6 · Main run

In [ ]:
# ── Main loop: per video encode once (+ shifted frames once), compress once; per question run every pass ──
import zlib, re
RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
ERR_PATH = os.path.join(OUT_DIR, "errors.log")
done = set()
if os.path.exists(RES_PATH):
    with open(RES_PATH) as f:
        for line in f:
            try:
                done.add(json.loads(line)["qid"])
            except Exception:
                pass
print(f"{len(done)} questions already in results (resume)")

_PREFIX = re.compile(r"^\s*\(?[A-Ha-h][\.\):]\s*")

def cycled_options(options):
    """Cyclic shift by one: new position j shows original option (j+1) % n, relabelled A, B, C, ..."""
    body = [_PREFIX.sub("", str(o)) for o in options]
    n = len(body)
    return [f"{LETTERS[j]}. {body[(j + 1) % n]}" for j in range(n)]

def map_back(res, n):
    """Express a cycled-order result in original option letters: original i sits at new position (i-1) % n."""
    lp_new = np.array(res["logp"])
    lp = np.array([lp_new[(i - 1) % n] for i in range(n)])
    srt = np.sort(lp)[::-1]
    out = dict(res)
    out.update(pred=LETTERS[int(lp.argmax())], logp=lp.round(4).tolist(), margin=float(srt[0] - srt[1]))
    return out

def run_question(Xf, Xs, comp, row):
    n_opt = min(len(row.options), len(LETTERS))
    suf = build_suffix_ids(row.question, row.options)
    T, L, D = Xf.shape
    def finish(res):
        res["correct"] = bool(res["pred"] == row.answer)
        return res
    rec = {"qid": row.qid, "video": row.video_key, "duration": row.duration, "task_type": row.task_type,
           "gt": row.answer, "n_full": T * L}
    rec["full"] = finish(llm_pass(Xf.reshape(-1, D), np.arange(T * L), T * L, suf, n_opt))
    rec["noise"] = {
        "frame_shift": finish(llm_pass(Xs.reshape(-1, D), np.arange(T * L), T * L, suf, n_opt)),
        "option_cycle": finish(map_back(llm_pass(Xf.reshape(-1, D), np.arange(T * L), T * L,
                                                 build_suffix_ids(row.question, cycled_options(row.options)), n_opt),
                                        n_opt)),
    }
    rec["methods"] = {}
    for (m, s), c in comp.items():
        res = finish(llm_pass(c["tokens"], c["pos"], c["n_full"], suf, n_opt))
        res["retention"] = round(c["tokens"].shape[0] / (T * L), 4)
        rec["methods"].setdefault(m, {})[s] = res
    return rec

table = load_videomme_table()
T_START = time.time()
time_left = lambda: MAX_RUNTIME_HOURS * 3600 - (time.time() - T_START)
n_new, n_err, consec_fail, CHECK_DONE, stop = 0, 0, 0, False, False

def _fail_guard():
    if consec_fail >= 6:
        raise RuntimeError("6 questions failed in a row — stopping instead of wasting GPU hours. Send me the log.")

for ch in VIDEOMME_CHUNKS:
    if stop or time_left() < 1800:
        print("⏱️ Time budget reached — skipping remaining chunks"); break
    vids = fetch_chunk(ch)
    sub = table[table.video_key.isin(vids.keys()) | table.video_id.isin(vids.keys())]
    sub = sub[~sub.qid.isin(done)]
    if MAX_Q_PER_CHUNK:
        sub = sub.sample(n=min(MAX_Q_PER_CHUNK, len(sub)), random_state=SEED)
    print(f"Chunk {ch}: {len(sub)} questions to run | durations: {sub.duration.value_counts().to_dict()}")
    for vkey, group in sub.groupby("video_key", sort=False):
        if time_left() < 900:
            stop = True; print("⏱️ Time budget reached — stopping"); break
        path = vids.get(vkey) or vids.get(group.video_id.iloc[0])
        t0 = time.time()
        try:
            frames = read_frames(path, NUM_FRAMES)
            if not CHECK_DONE:
                CHECK_DONE = True
                equivalence_check(frames, group.iloc[0])
            Xf, A, G, VA, VM = encode_video(preprocess(frames))
            Xs = encode_video(preprocess(read_frames(path, NUM_FRAMES, shift=NOISE_FRAME_SHIFT)))[0]
            comp = compress_all(Xf, A, G, VA, VM, seed=zlib.crc32(str(vkey).encode()))
        except RuntimeError as e:
            if "Equivalence check" in str(e):
                raise
            n_err += len(group); consec_fail += len(group)
            with open(ERR_PATH, "a") as f: f.write(f"{vkey}\tencode\t{type(e).__name__}: {e}\n")
            print(f"  ✗ video {vkey}: {type(e).__name__}: {e}"); _fail_guard(); continue
        except Exception as e:
            n_err += len(group); consec_fail += len(group)
            with open(ERR_PATH, "a") as f: f.write(f"{vkey}\tencode\t{type(e).__name__}: {e}\n")
            print(f"  ✗ video {vkey}: {type(e).__name__}: {e}"); _fail_guard(); continue
        for row in group.itertuples():
            try:
                rec = run_question(Xf, Xs, comp, row)
            except torch.cuda.OutOfMemoryError as e:
                torch.cuda.empty_cache(); n_err += 1; consec_fail += 1
                with open(ERR_PATH, "a") as f: f.write(f"{row.qid}\tOOM\t{e}\n")
                print(f"  ✗ {row.qid}: out of GPU memory (skipped)"); _fail_guard(); continue
            except Exception as e:
                n_err += 1; consec_fail += 1
                with open(ERR_PATH, "a") as f: f.write(f"{row.qid}\t{type(e).__name__}\t{e}\n")
                print(f"  ✗ {row.qid}: {type(e).__name__}: {e}"); _fail_guard(); continue
            with open(RES_PATH, "a") as f:
                f.write(json.dumps(rec) + "\n")
            done.add(row.qid); n_new += 1; consec_fail = 0
            full = rec["full"]["pred"]
            flips = sum(r["pred"] != full for m in rec["methods"].values() for r in m.values())
            nz = rec["noise"]
            print(f"  ✓ {row.qid} [{row.duration}] gt={row.answer} full={full} "
                  f"shift={nz['frame_shift']['pred']} cycle={nz['option_cycle']['pred']} "
                  f"| {flips}/{sum(len(v) for v in rec['methods'].values())} compressed passes changed "
                  f"| done {len(done)} | {time.time()-t0:.0f}s | {time_left()/3600:.1f} h left")
        del Xf, Xs, A, G, VA, VM, comp
        gc.collect(); torch.cuda.empty_cache()
    cleanup_chunk(ch)
print(f"\nFinished: {n_new} new questions, {n_err} errors, {len(done)} total in {RES_PATH}")


### 7 · Gate verdicts, noise floor, first look at flips

In [ ]:
# ── Step 1 analysis: gate verdicts, noise floor, first look at flips ──
import matplotlib.pyplot as plt

RES_PATH = os.path.join(OUT_DIR, "results.jsonl")
R = []
with open(RES_PATH) as f:
    for line in f:
        try:
            R.append(json.loads(line))
        except Exception:
            pass
n = len(R)
assert n >= 10, f"Only {n} results — nothing to analyse yet."
rng = np.random.default_rng(SEED)
vid = np.array([r["video"] for r in R])
videos = np.unique(vid)
v_rows = [np.flatnonzero(vid == v) for v in videos]
full_pred = np.array([r["full"]["pred"] for r in R])
full_cor = np.array([r["full"]["correct"] for r in R])
full_lp = [np.array(r["full"]["logp"]) for r in R]
B = 2000
boot_idx = [np.concatenate([v_rows[j] for j in rng.integers(0, len(videos), len(videos))]) for _ in range(B)]

def kl(p_lp, q_lp):
    p = np.exp(p_lp)
    return float((p * (p_lp - q_lp)).sum())

def stats(get):
    pred = np.array([get(r)["pred"] for r in R]); cor = np.array([get(r)["correct"] for r in R])
    rel = 100 * cor.mean() / full_cor.mean()
    rel_b = np.array([100 * cor[i].mean() / max(full_cor[i].mean(), 1e-9) for i in boot_idx])
    flip = pred != full_pred
    flip_b = np.array([flip[i].mean() for i in boot_idx])
    return {
        "acc": float(cor.mean()), "rel_acc": float(rel),
        "rel_ci": [float(np.percentile(rel_b, 2.5)), float(np.percentile(rel_b, 97.5))],
        "flip": float(flip.mean()), "flip_ci": [float(np.percentile(flip_b, 2.5)), float(np.percentile(flip_b, 97.5))],
        "harmful": float(((~cor) & full_cor).mean()), "helpful": float((cor & ~full_cor).mean()),
        "kl": float(np.mean([kl(full_lp[k], np.array(get(r)["logp"])) for k, r in enumerate(R)])),
        "flops_ratio": float(np.mean([get(r)["flops"] for r in R]) / np.mean([r["full"]["flops"] for r in R])),
        "time_ratio": float(np.mean([get(r)["sec"] for r in R]) / np.mean([r["full"]["sec"] for r in R])),
        "peak_gb": float(np.max([get(r).get("peak_gb", np.nan) for r in R])),
    }

print(f"=== {n} questions, {len(videos)} videos | full-token accuracy {full_cor.mean():.3f} "
      f"| full peak memory {np.max([r['full'].get('peak_gb', np.nan) for r in R]):.1f} GB ===\n")

# ---------------- noise floor ----------------
noise = {k: stats(lambda r, k=k: r["noise"][k]) for k in ["frame_shift", "option_cycle"]}
print("Noise floor (full model vs itself):")
for k, s in noise.items():
    print(f"   {k:13s} flips {s['flip']:.3f} (95% CI {s['flip_ci'][0]:.3f}–{s['flip_ci'][1]:.3f}) "
          f"| rel acc {s['rel_acc']:.1f}% | harmful/helpful {s['harmful']:.3f}/{s['helpful']:.3f} | KL {s['kl']:.3f}")

# ---------------- methods + gate ----------------
rows = []
for m, settings in R[0]["methods"].items():
    for s in settings:
        st = stats(lambda r, m=m, s=s: r["methods"][m][s])
        st["retention"] = float(np.mean([r["methods"][m][s]["retention"] for r in R]))
        ref = GATE_REFS.get((m, s))
        if ref is None:
            verdict, ref_txt = "no reference", ""
        else:
            lo, hi = min(ref["values"]), max(ref["values"])
            near = (lo - GATE_TOL_POINTS) <= st["rel_acc"] <= (hi + GATE_TOL_POINTS)
            covers = st["rel_ci"][0] <= hi and st["rel_ci"][1] >= lo
            verdict = "PASS" if (near or covers) else "FAIL"
            ref_txt = "/".join(f"{v:.1f}" for v in ref["values"])
        rows.append({"method": m, "setting": s, "retention": round(st["retention"], 4),
                     "acc": round(st["acc"], 4), "rel_acc_%": round(st["rel_acc"], 1),
                     "rel_ci_%": f"{st['rel_ci'][0]:.1f}–{st['rel_ci'][1]:.1f}", "published_%": ref_txt,
                     "gate": verdict, "flip": round(st["flip"], 4),
                     "flip_ci": f"{st['flip_ci'][0]:.3f}–{st['flip_ci'][1]:.3f}",
                     "harmful": round(st["harmful"], 4), "helpful": round(st["helpful"], 4),
                     "kl": round(st["kl"], 4), "flops_ratio": round(st["flops_ratio"], 4),
                     "time_ratio": round(st["time_ratio"], 4), "peak_gb": round(st["peak_gb"], 2)})
gate = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\nMethods (relative accuracy = method / full on the same questions):")
print(gate[["method", "setting", "retention", "rel_acc_%", "rel_ci_%", "published_%", "gate",
            "flip", "harmful", "helpful", "kl", "time_ratio"]].to_string(index=False))
gate.to_csv(os.path.join(OUT_DIR, "gate_table.csv"), index=False)

method_verdict = {}
for m, g in gate.groupby("method"):
    v = set(g.gate) - {"no reference"}
    method_verdict[m] = "no reference (baseline)" if not v else ("PASS" if v == {"PASS"} else "FAIL")
print("\nGate per method:", method_verdict)

# ---------------- exploratory: flips vs noise floor (pilot data) ----------------
nf = max(noise["frame_shift"]["flip"], noise["option_cycle"]["flip"])
parity = gate[gate["rel_ci_%"].apply(lambda t: float(t.split("–")[0]) <= 100 <= float(t.split("–")[1]))]
print(f"\nExploratory (pilot data, not a test of H1–H5): {len(parity)} settings at accuracy parity; "
      f"their flip rates {', '.join(f'{a}@{b}: {c:.3f}' for a, b, c in parity[['method','setting','flip']].values)}; "
      f"largest noise-floor flip rate {nf:.3f}")

# ---------------- plot ----------------
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
colors = {"fastvid": "tab:blue", "visionzip": "tab:orange", "dycoke": "tab:green", "uniform": "tab:gray", "random": "tab:red"}
for m, g in gate.groupby("method"):
    ax[0].plot(g.retention, g["rel_acc_%"], "o-", color=colors.get(m), label=m)
    ax[1].plot(g.retention, g.flip, "o-", color=colors.get(m), label=m)
for (m, s), ref in GATE_REFS.items():
    r = gate[(gate.method == m) & (gate.setting == s)]
    if len(r):
        for v in ref["values"]:
            ax[0].plot(r.retention.iloc[0], v, "x", color=colors.get(m), ms=9)
ax[0].axhline(100, c="k", ls="--", lw=0.8); ax[0].set_xscale("log")
ax[0].set_xlabel("token retention"); ax[0].set_ylabel("relative accuracy (%)")
ax[0].set_title("Relative accuracy (o = ours, x = published)"); ax[0].legend(fontsize=8)
for k, ls in [("frame_shift", ":"), ("option_cycle", "--")]:
    ax[1].axhline(noise[k]["flip"], c="k", ls=ls, lw=1, label=f"noise floor: {k}")
ax[1].set_xscale("log"); ax[1].set_xlabel("token retention"); ax[1].set_ylabel("answers changed vs full")
ax[1].set_title("Answer flips"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "step1.png"), dpi=150); plt.show()

summary = {"n": n, "n_videos": int(len(videos)), "model": MODEL_ID, "frames": NUM_FRAMES,
           "full_accuracy": float(full_cor.mean()), "noise_floor": noise,
           "gate_per_method": method_verdict, "table": rows}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
if QUICK_TEST:
    print("\n(Quick test: numbers are meaningless — this only proves the pipeline works.)")
print("Files saved in", OUT_DIR)


## What to send back
`summary.json`, `gate_table.csv`, `step1.png`, `gate_rules.json`, `equivalence_check.json` (and `errors.log` if not empty).
Methods that fail the gate are excluded from the main study and reported as such; the noise-floor numbers here are a first look only (pilot data).
